In [ ]:
# DONE jeżeli tresc misji sie zmienila i jest usuwana w dbo.MISJE, to niech sie usuwa/archiwizuje rowniez z dbo.MISJE_PODSUMOWANIE (jeżeli jest), wtedy skrypt będzie dzialal tak, ze jezeli brakuje jakiejs misji w tej tabeli, to wygeneruje podsumowanie od nowa

# DONE 1. pobieram treść misji
# DONE 2. wysyla ja do LLM
# DONE 3. w AI_LOGS (zmienic by był nowy krok) dodaje logi
# DONE 4. dostaje podsumowanie misji i wrzucam ja do tabeli

In [2]:
import sys
from pathlib import Path

ROOT = Path(r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl")
sys.path.insert(0, str(ROOT))

In [3]:
from moduly.utils import hash_do_wsad_json, sklej_warunki_w_WHERE
from moduly.db_core import utworz_engine_do_db
from moduly.ai_modele import llm_quest_summary
from moduly.ai_prompty_misje import get_quest_summary
from sqlalchemy import text
from moduly.ai_logi import (
    create_logs,
    save_ai_logs_to_db
)
import time

In [4]:
silnik = utworz_engine_do_db()

In [5]:
q_select_misja_hash = text(f"""
WITH MAIN AS (
	SELECT 
		M.MISJA_ID_MOJE_PK, ZM.HTML_SKOMPRESOWANY,
		ROW_NUMBER() OVER (PARTITION BY M.MISJA_ID_MOJE_PK ORDER BY DATA_WYSCRAPOWANIA DESC) AS RNK
	FROM dbo.MISJE AS M
	INNER JOIN dbo.ZRODLO_MISJE AS ZM
	  ON M.MISJA_ID_MOJE_PK = ZM.MISJA_ID_MOJE_FK
	WHERE 1=1
	  {sklej_warunki_w_WHERE(
          fabula=""
      )}
)

SELECT MISJA_ID_MOJE_PK, HTML_SKOMPRESOWANY
FROM MAIN
WHERE RNK = 1
""")

In [6]:
q_insert_summary = text("""
INSERT INTO dbo.MISJE_PODSUMOWANIA (
    MISJA_ID_MOJE_FK,
    PODSUMOWANIE
)
SELECT
    :misja_id_moje_fk,
    :podsumowanie
WHERE NOT EXISTS (
    SELECT 1
    FROM dbo.MISJE_PODSUMOWANIA
    WHERE MISJA_ID_MOJE_FK = :misja_id_moje_fk
)
""")

In [7]:
with silnik.connect() as conn:
    lista_krotek = conn.execute(q_select_misja_hash, {"fabula_en": "Path of de Hash'ey"}).all()

lista_krotek = [lista_krotek[0]]

In [8]:
llm = llm_quest_summary()

In [9]:
summary = {}
for misja_id, hash in lista_krotek:
    started_at = time.perf_counter()

    tresc_misji = hash_do_wsad_json(hash)
    answer = get_quest_summary(llm=llm, mission=tresc_misji)
    summary[misja_id] = answer.content[1].get("text", "")

    logs = create_logs(
        raw_response=answer,
        llm=llm,
        misja_id_moje_fk=misja_id,
        stage="quest_summary",
        duration_ms=round((time.perf_counter() - started_at) * 1000),
        input_chars=len(tresc_misji),
        output_chars=len(answer.content[1].get("text", ""))
        )
    save_ai_logs_to_db(silnik=silnik, logs=logs)


In [10]:
with silnik.begin() as conn:
    conn.execute(q_insert_summary, [
        {"misja_id_moje_fk": misja_id, "podsumowanie": podsumowanie}
        for misja_id, podsumowanie in summary.items()
    ])